In [1]:
from Dataloader_WG import read_wg_data_single
from model import fNIRS_T

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score
import pandas as pd

class LabelSmoothing(torch.nn.Module):
	def __init__(self, smoothing=0.0):
		super(LabelSmoothing, self).__init__()
		self.confidence = 1.0 - smoothing
		self.smoothing = smoothing

	def forward(self, x, target):
		logprobs = torch.nn.functional.log_softmax(x, dim=-1)
		nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
		nll_loss = nll_loss.squeeze(1)
		smooth_loss = -logprobs.mean(dim=-1)
		loss = self.confidence * nll_loss + self.smoothing * smooth_loss
		return loss.mean()

class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.sampling_points = config['sampling_points']
		self.flooding_level = config['flooding_level']
		self.metric_epochs = config['metric_epochs']
		self.eeg_root = config['eeg_root']
		self.nirs_root = config['nirs_root']
		self.baseline = config['baseline']

		self.model = fNIRS_T(n_class=2, sampling_point=self.sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(self.device)
		self.criterion = LabelSmoothing(0.1)
		self.optimizer = torch.optim.AdamW(self.model.parameters())
		lrStep = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=30)

		os.makedirs('Results', exist_ok=True)
	
	def train_epoch(self, epoch, train_loader):
		self.model.train()
		total_correct, total_loss = 0, 0

		for batch_nirs, batch_labels in train_loader:
			batch_nirs = batch_nirs.to(self.device, dtype=torch.float32)
			batch_labels = batch_labels.to(self.device)
			
			outputs = self.model(batch_nirs)
			loss = self.criterion(outputs, batch_labels.long())

			if epoch < 30:
				b = self.flooding_level[0]
			elif epoch < 50:
				b = self.flooding_level[1]
			else:
				b = self.flooding_level[2]

			loss = (loss - b).abs() + b
			self. optimizer.zero_grad()
			loss.backward()
			self.optimizer.step()
			
			preds = torch.argmax(outputs, dim=1)
			total_correct += (preds == batch_labels).sum().item()
			total_loss += loss.item()

			train_loss = total_loss / len(train_loader)
			train_acc = total_correct / len(train_loader.dataset)
			# print(f"Training Loss: {train_loss:.2f} | Training Acc: {train_acc:.2f}")
			
			return train_loss, train_acc
	
	def evaluate_epoch(self, epoch, eval_loader):
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_nirs, eval_labels in eval_loader:
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
				eval_labels = eval_labels.to(self.device)
				
				outputs = self.model(eval_nirs)
				loss = self.criterion(outputs, eval_labels.long())
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				total_loss += loss.item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds, zero_division=0)
		f1 = f1_score(all_labels, all_preds, zero_division=0)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		# print(f"Evaluation Loss: {loss:.2f} | Evaluation Acc: {acc:.2f}")
		return loss, acc, precision, recall, f1, kappa

	def train_subject(self, subject_id, mode):
		seed= 42
		torch.manual_seed(seed)
		_, nirs, labels = read_wg_data_single(subject_id, self.eeg_root, self.nirs_root, self.baseline)
		nirs, labels = nirs.to(self.device), labels.to(self.device)
		
		train_size = int(0.8 * len(labels)) # training/testing ratio
		eval_size = len(labels) - train_size
		
		dataset = torch.utils.data.TensorDataset(nirs, labels)
		train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
		train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		
		# Reset codebook variables for new subject
		# self.init_codebook_vars(self.config['dict_len'], self.config['emb_size'])
		
		for epoch in range(self.config['num_epochs']):
			train_loss, train_acc = self.train_epoch(epoch, train_loader)
			eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(epoch, eval_loader)
			
			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject_id} | Epoch {epoch+1}/{self.config['num_epochs']} | "
				f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | "
				f"Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")

		mean_acc, std_acc = np.mean(acc_list[-self.metric_epochs:]), np.std(acc_list[-self.metric_epochs:])
		mean_precision, std_precision = np.mean(precision_list[-self.metric_epochs:]), np.std(precision_list[-self.metric_epochs:])
		mean_recall, std_recall = np.mean(recall_list[-self.metric_epochs:]), np.std(recall_list[-self.metric_epochs:])
		mean_f1, std_f1 = np.mean(f1_list[-self.metric_epochs:]), np.std(f1_list[-self.metric_epochs:])
		mean_kappa = np.mean(kappa_list[-self.metric_epochs:])

		mean_acc, std_acc = mean_acc * 100, std_acc * 100
		mean_precision, std_precision = mean_precision * 100, std_precision * 100
		mean_recall, std_recall = mean_recall * 100, std_recall * 100
		mean_f1, std_f1 = mean_f1 * 100, std_f1 * 100

		log_path = f'Results/log_s{subject_id:2d}.txt'
		with open(log_path, 'a') as log_file:
			log_file.write(f'Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')
			log_file.write(f'Precision: {mean_precision:.2f} ± {std_precision:.2f}\n')
			log_file.write(f'Recall: {mean_recall:.2f} ± {std_recall:.2f}\n')
			log_file.write(f'F1: {mean_f1:.2f} ± {std_f1:2f}\n')
			log_file.write(f'Kappa: {mean_kappa:.2f}\n')
			log_file.write('\n')

		log_excel = 'Results/Log.xlsx'
		new_row = pd.DataFrame([[mean_acc, std_acc, mean_precision, std_precision, mean_recall, std_recall, mean_f1, std_f1, mean_kappa]], 
							columns=['Accuracy', 'Std_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
		new_row = new_row.round(2)
		if not os.path.exists(log_excel):
			new_row.to_excel(log_excel, index=False, engine='openpyxl')
		else:
			with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
				if 'Sheet1' in writer.sheets:
					startrow = writer.sheets['Sheet1'].max_row
				else:
					startrow = 0
				new_row.to_excel(writer, index=False, header=False, startrow=startrow)

		print(f'Subject:{subject_id} Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')			
		return

# Hyperparameters
config = {
	'depth': 4,
	'query_size': 64,
	'key_size': 64,
	'value_size': 64,
	'emb_size': 64,
	'dict_len': 512,
	'decay': 0.99,
	'num_heads': 4,
	'expansion': 2,
	'conv_dropout': 0.3,
	'self_dropout': 0.3,
	'cross_dropout': 0.3,
	'cls_dropout': 0.5,
	'num_classes': 2,
	'batch_size': 32,
	'num_epochs': 100,
	'metric_epochs': 20,
	'learning_rate': 1e-4,
	'sampling_points': 100,
	'flooding_level': [0.45, 0.40, 0.35],
	'eeg_root': '../../Data/EF-WG/Raw/',
	'nirs_root': '../../Data/EF-WG/Raw/',
	'baseline': False
}

# Initialize and run trainer
trainer = Trainer(config)
for subject in range(26):
	subject += 1
	print(f"\n=== Subject {subject} ===")
	results = trainer.train_subject(subject, mode=0) # 0 = MI, 1 = MA

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



=== Subject 1 ===
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks shape (100, 36, 2)
tasks 